# Walkthrough: see each step working on real data

The client (Pepsi) gave the ad agency (GroupM) a budget. The ad agency bought ads from
publishers (Instagram, Facebook, YouTube, Hulu, TikTok), and they send bills. This
notebook shows each finished step of the checking system working on the sample data
in `data/sample/`.

**How to run:** in the project folder run `uv run jupyter lab`, open this file, then
**Run → Run All Cells**.

**Rules for this notebook**
- It only **calls** code from `src/invoice_recon/`. The real logic lives there, where it is tested.
- It never reads `answer_key.csv`.
- One section per finished step.

In [1]:
import json
from decimal import Decimal

import pandas as pd
from pydantic import ValidationError

from invoice_recon.config import get_settings
from invoice_recon.models import ContractTerms, InvoiceLine, PlanLine, Result

pd.set_option("display.max_colwidth", None)
settings = get_settings()
print("Data folder:", settings.data_folder.relative_to(settings.data_folder.parents[1]))

Data folder: data/sample


---
# Step 1.1: The 5 forms

Every record must fit its form (`src/invoice_recon/models.py`). A record that doesn't
fit is **rejected the moment it is created**, before it can reach the money math.

## 0. The 5 blank forms

Before anything is filled in, here are the 5 empty forms: every field, what kind of
value it holds, and its rules. These tables are produced **from the code in
`models.py` itself**, so they always match what the system really checks.

In [2]:
from IPython.display import Markdown, display

from invoice_recon.describe import blank_form
from invoice_recon.models import DeliveryRecord

for form in (PlanLine, DeliveryRecord, InvoiceLine, ContractTerms, Result):
    display(Markdown(f"### {form.__name__}"))
    display(pd.DataFrame(blank_form(form)))

### PlanLine

,field,kind of value,required,rules,meaning
0,plan_line_id,text,yes,can't be empty,"Unique ID of this plan row, e.g. PL-001."
1,purchase_order_number,text,yes,can't be empty,"Covers a whole campaign at one publisher, not a single plan line."
2,publisher,text,yes,can't be empty,"Who runs the ad, e.g. Instagram."
3,description,text,yes,can't be empty,"What is being bought: campaign, publisher and ad type."
4,start_date,date,yes,-,First day the ad runs.
5,end_date,date,yes,-,Last day the ad runs. Not before start_date.
6,planned_impressions,whole number,yes,>= 0,Views ordered for the whole run.
7,price_per_1000_views,exact decimal,yes,"never a float, >= 0, at most 4 decimal places","Agreed price for every 1,000 views (CPM)."
8,planned_cost,exact decimal,yes,"never a float, >= 0, at most 2 decimal places","Must equal views x price per view, to the cent."
9,status,"one of: active, cancelled",yes,-,"Active, or cancelled from cancelled_effective_date."


### DeliveryRecord

,field,kind of value,required,rules,meaning
0,plan_line_id,text,yes,can't be empty,Which plan line these views belong to.
1,publisher,text,yes,can't be empty,Must be the same publisher as on the plan line.
2,month,text,yes,month as YYYY-MM,Which month the views were delivered in.
3,publisher_count,whole number,yes,>= 0,"Views counted by the publisher (Instagram, Facebook)."
4,ad_server_count,whole number,yes,>= 0,Views counted by the ad agency's (GroupM's) own ad server.


### InvoiceLine

,field,kind of value,required,rules,meaning
0,invoice_number,text,yes,can't be empty,"The bill's number, e.g. IG-INV-2026-0142."
1,invoice_line_number,whole number,yes,>= 1,"Line position on the bill, starting at 1."
2,publisher,text,yes,can't be empty,Who sent the bill.
3,service_month,text,yes,month as YYYY-MM,Which month this line charges for.
4,purchase_order_number,text,no,can't be empty,Missing on about a quarter of lines.
5,line_description,text,yes,can't be empty,"The publisher's own wording, e.g. ""PZS | Reels | JAN26""."
6,billed_impressions,whole number,yes,>= 0,Views the publisher is charging for.
7,rate_per_1000_views,exact decimal,yes,"never a float, >= 0, at most 4 decimal places","Price per 1,000 views used on the bill."
8,amount,exact decimal,yes,"never a float, >= 0, at most 2 decimal places",Money charged. May be wrong; checked later.
9,source_file,text,yes,can't be empty,The bill file this line was read from.


### ContractTerms

,field,kind of value,required,rules,meaning
0,publisher,text,yes,can't be empty,The publisher these rules apply to.
1,agency,text,yes,can't be empty,"The ad agency that pays, e.g. GroupM."
2,client,text,yes,can't be empty,"The brand the ads are for, e.g. Pepsi."
3,bill_on,"one of: publisher_count, ad_server_count",yes,-,Whose view count is used for billing.
4,allowed_difference_percent,exact decimal,yes,"never a float, >= 0, <= 100",How far the publisher's count may be above the ad server count.
5,pay_for_extra_views_up_to_percent,exact decimal,yes,"never a float, >= 0, <= 100",0 = extra views are never paid. 10 = up to 10% extra views are paid.


### Result

,field,kind of value,required,rules,meaning
0,invoice_number,text,yes,can't be empty,The bill this result is for.
1,invoice_line_number,whole number,yes,>= 1,The line on that bill.
2,matched_plan_line_id,text,no,can't be empty,None when no plan line matches.
3,match_confidence,number,yes,">= 0, <= 1",How sure the match is: 1.0 = certain.
4,expected_amount,exact decimal,yes,"never a float, >= 0, at most 2 decimal places",What the ad agency (GroupM) should pay under the contract.
5,invoiced_amount,exact decimal,yes,"never a float, >= 0, at most 2 decimal places",What the publisher billed.
6,cause,"one of: none, over_delivery_billed, over_delivery_within_contract, over_delivery_beyond_contract, outside_flight_dates, duplicate_line, count_mismatch_beyond_tolerance, billed_on_wrong_count, cancelled_line_billed, math_error, rate_mismatch, no_matching_order",yes,-,"Why the bill differs, from the fixed list of causes."
7,evidence,list of text,no,-,"Short facts that prove the cause, each naming its source."
8,decision,"one of: approve, pay_less, dispute, send_to_human",yes,-,"What to do: approve, pay less, dispute or send to a human."


**Rules that link fields together** (also checked by the forms):
- `PlanLine`: end date not before start date · a cancelled line needs a cancel date ·
  planned cost = views × price per view, to the cent
- `Result`: the difference (billed − expected) is worked out by the form itself ·
  a line with a difference can't have cause `none`

## 1. Real data: the 5 contracts become `ContractTerms` records

`contracts.json` holds the billing rules the ad agency (GroupM) agreed with each
publisher. Each one is checked by the `ContractTerms` form.

In [3]:
raw = json.loads((settings.data_folder / "contracts.json").read_text())
contracts = [ContractTerms(**c) for c in raw]

print(f"{len(contracts)} of {len(raw)} contracts fit the form\n")
pd.DataFrame([c.model_dump(mode="json") for c in contracts])

5 of 5 contracts fit the form



,publisher,agency,client,bill_on,allowed_difference_percent,pay_for_extra_views_up_to_percent
0,Instagram,GroupM,Pepsi,publisher_count,10,0
1,Facebook,GroupM,Pepsi,publisher_count,10,0
2,YouTube,GroupM,Pepsi,ad_server_count,10,0
3,Hulu,GroupM,Pepsi,ad_server_count,10,10
4,TikTok,GroupM,Pepsi,publisher_count,5,0


**How to read it:** Hulu is the only publisher that pays for extra views (up to 10%).
YouTube and Hulu bill on the ad agency's (GroupM's) own ad server count; the others on
their own count, as long as it is no more than 10% (TikTok: 5%) above the ad server count.

## 2. A small-number plan line

100 views at \$1 each. The industry writes the price per 1,000 views, so \$1 per view is
\$1,000 per 1,000 views. 100 views cost **\$100**.

In [4]:
plan = PlanLine(
    plan_line_id="PL-001",
    purchase_order_number="PO-5000",
    publisher="Instagram",
    description="Pepsi Zero Sugar Big Game - Instagram Stories video",
    start_date="2026-01-01",
    end_date="2026-03-31",
    planned_impressions=100,
    price_per_1000_views=Decimal("1000"),
    planned_cost=Decimal("100.00"),
    status="active",
)
pd.Series(plan.model_dump(mode="json"), name="value").to_frame()

,value
plan_line_id,PL-001
purchase_order_number,PO-5000
publisher,Instagram
description,Pepsi Zero Sugar Big Game - Instagram Stories video
start_date,2026-01-01
end_date,2026-03-31
planned_impressions,100
price_per_1000_views,1000
planned_cost,100.00
status,active


## 3. Bad values are rejected, each with its exact reason

We take the good plan line above and change one thing at a time.

In [5]:
good = plan.model_dump()
attempts = {
    "views = -100": {"planned_impressions": -100},
    "ends before it starts": {"end_date": "2025-12-31"},
    "status = 'maybe'": {"status": "maybe"},
    "cancelled, but no cancel date": {"status": "cancelled"},
    "cost $99, but 100 views x $1 = $100": {"planned_cost": Decimal("99.00")},
    "money as a float (100.0)": {"planned_cost": 100.0},
    "an unknown field 'colour'": {"colour": "blue"},
    "nothing wrong": {},
}

rows = []
for what, change in attempts.items():
    try:
        PlanLine(**{**good, **change})
        result = "✅ accepted"
    except ValidationError as error:
        result = "❌ rejected: " + error.errors()[0]["msg"]
    rows.append({"what we tried": what, "result": result})

pd.DataFrame(rows)

,what we tried,result
0,views = -100,❌ rejected: Input should be greater than or equal to 0
1,ends before it starts,"❌ rejected: Value error, end_date is before start_date"
2,status = 'maybe',❌ rejected: Input should be 'active' or 'cancelled'
3,"cancelled, but no cancel date","❌ rejected: Value error, a cancelled line needs a cancelled_effective_date, and only then"
4,"cost $99, but 100 views x $1 = $100","❌ rejected: Value error, planned_cost 99.00 doesn't equal views x price = 100.00"
5,money as a float (100.0),"❌ rejected: Value error, money must be a Decimal or text like '12.50', never a float"
6,an unknown field 'colour',❌ rejected: Extra inputs are not permitted
7,nothing wrong,✅ accepted


## 4. A bill line, and the result for it

Instagram bills **107 views at \$1 = \$107**, but the contract only allows **\$100**.
The bill line form accepts it (bills *can* be wrong; finding that is a later step),
and the `Result` form works out the difference itself.

In [6]:
bill_line = InvoiceLine(
    invoice_number="IG-INV-2026-0142",
    invoice_line_number=1,
    publisher="Instagram",
    service_month="2026-01",
    purchase_order_number="PO-5000",
    line_description="PZS | IG Stories Vid | JAN26",
    billed_impressions=107,
    rate_per_1000_views=Decimal("1000"),
    amount=Decimal("107.00"),
    source_file="invoices/instagram_2026-01.pdf",
    source_location="page 1, row 1",
)

result = Result(
    invoice_number=bill_line.invoice_number,
    invoice_line_number=bill_line.invoice_line_number,
    matched_plan_line_id=plan.plan_line_id,
    match_confidence=1.0,
    expected_amount=Decimal("100.00"),
    invoiced_amount=bill_line.amount,
    cause="over_delivery_billed",
    evidence=("delivery: 107 views delivered, 100 ordered; contract pays 0% extra",),
    decision="pay_less",
)
pd.Series(result.model_dump(mode="json"), name="value").to_frame()

,value
invoice_number,IG-INV-2026-0142
invoice_line_number,1
matched_plan_line_id,PL-001
match_confidence,1.0
expected_amount,100.00
invoiced_amount,107.00
cause,over_delivery_billed
evidence,"[delivery: 107 views delivered, 100 ordered; contract pays 0% extra]"
decision,pay_less
difference,7.00


The difference is **\$7.00**. A difference with cause `"none"` is refused:

In [7]:
try:
    Result(**{**result.model_dump(exclude={"difference"}), "cause": "none"})
except ValidationError as error:
    print("❌ rejected:", error.errors()[0]["msg"])

❌ rejected: Value error, a line with a difference needs a cause other than 'none'


## 5. Why money is `Decimal`, never `float`

In [8]:
print("float:  ", 0.1 + 0.2)
print("Decimal:", Decimal("0.1") + Decimal("0.2"))

float:   0.30000000000000004
Decimal: 0.3


## 6. A record can't be changed after it is created

So a bill line can't silently change halfway through the checks.

In [9]:
try:
    plan.planned_cost = Decimal("1.00")
except ValidationError as error:
    print("❌ rejected:", error.errors()[0]["msg"])

❌ rejected: Instance is frozen


---
# Step 1.2: Load the clean files

The three clean files are loaded into their forms by `load_reference_data()` in
`src/invoice_recon/loaders.py`, then checked against each other.

| ① Input file (as stored) | ② Blank form | ③ Filled forms |
|---|---|---|
| `media_plan.xlsx` | `PlanLine` | 40 plan lines |
| `delivery/*.csv` | `DeliveryRecord` | 68 delivery records |
| `contracts.json` | `ContractTerms` | 5 contracts |

Then ④ six cross-file checks, and quick lookups for the later steps.

## ① Input files, exactly as they are stored

In [10]:
plan_path = settings.data_folder / "media_plan.xlsx"
plan_as_stored = pd.read_excel(plan_path)
print(f"media_plan.xlsx: first 5 of {len(plan_as_stored)} rows, as they look in Excel")
plan_as_stored.head()

media_plan.xlsx: first 5 of 40 rows, as they look in Excel


,plan_line_id,purchase_order_number,publisher,description,start_date,end_date,planned_impressions,price_per_1000_views,planned_cost,status,cancelled_effective_date
0,PL-001,PO-5000,Instagram,Pepsi Zero Sugar Big Game - Instagram Stories video,2026-01-01,2026-03-31,3824000,8.25,31548.0,active,NaT
1,PL-002,PO-5000,Instagram,Pepsi Zero Sugar Big Game - Instagram Reels video,2026-01-01,2026-03-31,6506000,9.50,61807.0,active,NaT
2,PL-003,PO-5000,Instagram,Pepsi Zero Sugar Big Game - Instagram Feed image,2026-01-01,2026-03-31,4286000,12.50,53575.0,active,NaT
3,PL-004,PO-5000,Instagram,Pepsi Zero Sugar Big Game - Instagram Feed carousel,2026-01-01,2026-01-31,10935000,8.50,92947.5,active,NaT
4,PL-005,PO-5001,Instagram,Pepsi Wild Cherry Spring - Instagram Stories video,2026-01-01,2026-03-31,8912000,8.25,73524.0,active,NaT


In [11]:
delivery_path = settings.data_folder / "delivery" / "delivery_2026-01.csv"
print("delivery_2026-01.csv: first 5 lines of the raw text\n")
print("\n".join(delivery_path.read_text().splitlines()[:5]))

delivery_2026-01.csv: first 5 lines of the raw text

plan_line_id,publisher,month,publisher_count,ad_server_count
PL-001,Instagram,2026-01,1475215,1453086
PL-002,Instagram,2026-01,2204466,2139028
PL-003,Instagram,2026-01,1447817,1402315
PL-004,Instagram,2026-01,10678674,10504348


In [12]:
contracts_text = (settings.data_folder / "contracts.json").read_text()
print("contracts.json: the first contract, as raw text\n")
print("\n".join(contracts_text.splitlines()[:10]))

contracts.json: the first contract, as raw text

[
  {
    "publisher": "Instagram",
    "bill_on": "publisher_count",
    "allowed_difference_percent": 10,
    "pay_for_extra_views_up_to_percent": 0,
    "agency": "GroupM",
    "client": "Pepsi"
  },
  {


## ② The blank forms these files go into

In [13]:
for form in (PlanLine, DeliveryRecord, ContractTerms):
    display(Markdown(f"### {form.__name__}"))
    display(pd.DataFrame(blank_form(form)))

### PlanLine

,field,kind of value,required,rules,meaning
0,plan_line_id,text,yes,can't be empty,"Unique ID of this plan row, e.g. PL-001."
1,purchase_order_number,text,yes,can't be empty,"Covers a whole campaign at one publisher, not a single plan line."
2,publisher,text,yes,can't be empty,"Who runs the ad, e.g. Instagram."
3,description,text,yes,can't be empty,"What is being bought: campaign, publisher and ad type."
4,start_date,date,yes,-,First day the ad runs.
5,end_date,date,yes,-,Last day the ad runs. Not before start_date.
6,planned_impressions,whole number,yes,>= 0,Views ordered for the whole run.
7,price_per_1000_views,exact decimal,yes,"never a float, >= 0, at most 4 decimal places","Agreed price for every 1,000 views (CPM)."
8,planned_cost,exact decimal,yes,"never a float, >= 0, at most 2 decimal places","Must equal views x price per view, to the cent."
9,status,"one of: active, cancelled",yes,-,"Active, or cancelled from cancelled_effective_date."


### DeliveryRecord

,field,kind of value,required,rules,meaning
0,plan_line_id,text,yes,can't be empty,Which plan line these views belong to.
1,publisher,text,yes,can't be empty,Must be the same publisher as on the plan line.
2,month,text,yes,month as YYYY-MM,Which month the views were delivered in.
3,publisher_count,whole number,yes,>= 0,"Views counted by the publisher (Instagram, Facebook)."
4,ad_server_count,whole number,yes,>= 0,Views counted by the ad agency's (GroupM's) own ad server.


### ContractTerms

,field,kind of value,required,rules,meaning
0,publisher,text,yes,can't be empty,The publisher these rules apply to.
1,agency,text,yes,can't be empty,"The ad agency that pays, e.g. GroupM."
2,client,text,yes,can't be empty,"The brand the ads are for, e.g. Pepsi."
3,bill_on,"one of: publisher_count, ad_server_count",yes,-,Whose view count is used for billing.
4,allowed_difference_percent,exact decimal,yes,"never a float, >= 0, <= 100",How far the publisher's count may be above the ad server count.
5,pay_for_extra_views_up_to_percent,exact decimal,yes,"never a float, >= 0, <= 100",0 = extra views are never paid. 10 = up to 10% extra views are paid.


## ③ Fill the forms

One call loads all three files, fills the forms, and runs the cross-file checks.
If anything were wrong, it would stop and list every problem.

In [14]:
from invoice_recon.loaders import DataLoadError, load_reference_data

ref = load_reference_data(settings.data_folder)

print(
    f"{len(ref.plan_lines)} plan lines, {len(ref.deliveries)} delivery records, "
    f"{len(ref.contracts)} contracts"
)
print("✅ every record fits its form, and all 6 cross-file checks passed")

40 plan lines, 68 delivery records, 5 contracts
✅ every record fits its form, and all 6 cross-file checks passed


### Before and after: one row, from text to exact values

Every cell is read as **text** first. The form then turns it into the right kind of
value. Money goes straight from text to an exact `Decimal`, so it is never a float.

In [15]:
row_as_text = pd.read_excel(plan_path, dtype=str, keep_default_na=False).iloc[0]
pl_001 = ref.plan_lines["PL-001"]

pd.DataFrame(
    [
        {
            "field": field,
            "in the file (text)": repr(row_as_text[field]),
            "in the form": repr(getattr(pl_001, field)),
            "kind of value": type(getattr(pl_001, field)).__name__,
        }
        for field in PlanLine.model_fields
    ]
)

,field,in the file (text),in the form,kind of value
0,plan_line_id,'PL-001','PL-001',str
1,purchase_order_number,'PO-5000','PO-5000',str
2,publisher,'Instagram','Instagram',str
3,description,'Pepsi Zero Sugar Big Game - Instagram Stories video','Pepsi Zero Sugar Big Game - Instagram Stories video',str
4,start_date,'2026-01-01 00:00:00',"datetime.date(2026, 1, 1)",date
5,end_date,'2026-03-31 00:00:00',"datetime.date(2026, 3, 31)",date
6,planned_impressions,'3824000',3824000,int
7,price_per_1000_views,'8.25',Decimal('8.25'),Decimal
8,planned_cost,'31548',Decimal('31548'),Decimal
9,status,'active',<PlanStatus.ACTIVE: 'active'>,PlanStatus


### All 40 plan lines

In [16]:
pd.DataFrame([p.model_dump(mode="json") for p in ref.plan_lines.values()])

,plan_line_id,purchase_order_number,publisher,description,start_date,end_date,planned_impressions,price_per_1000_views,planned_cost,status,cancelled_effective_date
0,PL-001,PO-5000,Instagram,Pepsi Zero Sugar Big Game - Instagram Stories video,2026-01-01,2026-03-31,3824000,8.25,31548,active,NaN
1,PL-002,PO-5000,Instagram,Pepsi Zero Sugar Big Game - Instagram Reels video,2026-01-01,2026-03-31,6506000,9.5,61807,active,NaN
2,PL-003,PO-5000,Instagram,Pepsi Zero Sugar Big Game - Instagram Feed image,2026-01-01,2026-03-31,4286000,12.5,53575,active,NaN
3,PL-004,PO-5000,Instagram,Pepsi Zero Sugar Big Game - Instagram Feed carousel,2026-01-01,2026-01-31,10935000,8.5,92947.5,active,NaN
4,PL-005,PO-5001,Instagram,Pepsi Wild Cherry Spring - Instagram Stories video,2026-01-01,2026-03-31,8912000,8.25,73524,active,NaN
5,PL-006,PO-5001,Instagram,Pepsi Wild Cherry Spring - Instagram Reels video,2026-01-01,2026-03-31,3535000,9.25,32698.75,active,NaN
6,PL-007,PO-5001,Instagram,Pepsi Wild Cherry Spring - Instagram Feed image,2026-02-01,2026-03-31,10279000,11.5,118208.5,active,NaN
7,PL-008,PO-5001,Instagram,Pepsi Wild Cherry Spring - Instagram Feed carousel,2026-01-15,2026-03-15,11195000,9.25,103553.75,active,NaN
8,PL-009,PO-5011,Facebook,Pepsi Wild Cherry Spring - Facebook Feed video,2026-01-01,2026-03-31,13647000,10.25,139881.75,active,NaN
9,PL-010,PO-5011,Facebook,Pepsi Wild Cherry Spring - Facebook Feed image,2026-01-01,2026-03-31,9873000,7.25,71579.25,cancelled,2026-02-01


### Delivery records: the first 8 of 68

In [17]:
deliveries = pd.DataFrame([d.model_dump() for d in ref.deliveries.values()])
print("Records per month:", deliveries["month"].value_counts().sort_index().to_dict())
deliveries.head(8)

Records per month: {'2026-01': 35, '2026-02': 33}


,plan_line_id,publisher,month,publisher_count,ad_server_count
0,PL-001,Instagram,2026-01,1475215,1453086
1,PL-002,Instagram,2026-01,2204466,2139028
2,PL-003,Instagram,2026-01,1447817,1402315
3,PL-004,Instagram,2026-01,10678674,10504348
4,PL-005,Instagram,2026-01,2984159,2923247
5,PL-006,Instagram,2026-01,1169556,1143051
6,PL-008,Instagram,2026-01,3115518,3052928
7,PL-009,Facebook,2026-01,4577110,4445680


### The 5 contracts

In [18]:
pd.DataFrame([c.model_dump(mode="json") for c in ref.contracts.values()])

,publisher,agency,client,bill_on,allowed_difference_percent,pay_for_extra_views_up_to_percent
0,Instagram,GroupM,Pepsi,publisher_count,10,0
1,Facebook,GroupM,Pepsi,publisher_count,10,0
2,YouTube,GroupM,Pepsi,ad_server_count,10,0
3,Hulu,GroupM,Pepsi,ad_server_count,10,10
4,TikTok,GroupM,Pepsi,publisher_count,5,0


## ④ Quick lookups for the later steps

Matching and the money math will ask exactly these kinds of questions.

In [19]:
plan = ref.plan_lines["PL-001"]
print(
    f"Plan line PL-001: {plan.publisher}, {plan.planned_impressions:,} views "
    f"at ${plan.price_per_1000_views} per 1,000, cost ${plan.planned_cost:,}"
)

january = ref.delivery_for("PL-001", "2026-01")
print(
    f"PL-001 in January: {january.publisher_count:,} views by {january.publisher}, "
    f"{january.ad_server_count:,} by the ad server"
)

hulu = ref.contracts["Hulu"]
print(
    f"Hulu's contract: bill on {hulu.bill_on}, "
    f"pays up to {hulu.pay_for_extra_views_up_to_percent}% extra views"
)

print(
    "PL-007 in January:",
    ref.delivery_for("PL-007", "2026-01"),
    "(it starts on Feb 1, so nothing was delivered)",
)

Plan line PL-001: Instagram, 3,824,000 views at $8.25 per 1,000, cost $31,548
PL-001 in January: 1,475,215 views by Instagram, 1,453,086 by the ad server
Hulu's contract: bill on ad_server_count, pays up to 10% extra views
PL-007 in January: None (it starts on Feb 1, so nothing was delivered)


## What happens with a broken file

A **throwaway copy** of the three files gets three planted problems. The real files
in `data/sample/` are never touched.

1. PL-001's planned views set to −100
2. A delivery row for PL-999, which isn't in the plan
3. TikTok's contract removed

In [20]:
import json
import shutil
import tempfile
from pathlib import Path

import openpyxl

with tempfile.TemporaryDirectory() as folder:
    copy = Path(folder)
    shutil.copy(settings.data_folder / "media_plan.xlsx", copy)
    shutil.copy(settings.data_folder / "contracts.json", copy)
    shutil.copytree(settings.data_folder / "delivery", copy / "delivery")

    book = openpyxl.load_workbook(copy / "media_plan.xlsx")
    book.active.cell(2, 7, -100)  # row 2 = PL-001, column 7 = planned_impressions
    book.save(copy / "media_plan.xlsx")

    with (copy / "delivery" / "delivery_2026-01.csv").open("a") as f:
        f.write("PL-999,Instagram,2026-01,100,98\n")

    kept = [c for c in json.loads(contracts_text) if c["publisher"] != "TikTok"]
    (copy / "contracts.json").write_text(json.dumps(kept))

    try:
        load_reference_data(copy)
    except DataLoadError as error:
        print(error)

3 problem(s) in the data files:
- media_plan.xlsx row 2: planned_impressions: Input should be greater than or equal to 0
- delivery 2026-01: plan line PL-999 is not in the media plan
- contracts.json: no contract for publisher TikTok


Loading stops, and **all three problems are listed at once**, each with where it is.
PL-001's deliveries are *not* falsely reported as "not in the media plan": its row
exists in the file, it was just rejected.

---
# Step 1.4: Exact money handling

The money rules live in `src/invoice_recon/money.py` and are used wherever money is
calculated.

| ① Input | ② The rule | ③ Output |
|---|---|---|
| Exact math with many decimal places, e.g. \$12,170.52375 | Decimal only · round half up · whole cents · each line first | \$12,170.52 |

This section only shows the **rounding**. Working out what *should be paid* under the
contract is step 4.1.

## ① Input: exact math on real January deliveries

Views counted by the publisher × the plan's price per 1,000 views. The exact result
has many decimal places. The float version is shown too, for comparison.

In [21]:
from invoice_recon.money import cost_of_views, to_cents, total

rows = []
for (plan_id, _month), d in list(ref.deliveries.items())[:8]:
    price = ref.plan_lines[plan_id].price_per_1000_views
    rows.append(
        {
            "plan line": plan_id,
            "publisher": d.publisher,
            "views (publisher count)": f"{d.publisher_count:,}",
            "price per 1,000": f"${price}",
            "① exact math (Decimal)": str(Decimal(d.publisher_count) * price / 1000),
            "same math as float": repr(d.publisher_count * float(price) / 1000),
            "③ rounded to cents": f"${cost_of_views(d.publisher_count, price):,}",
        }
    )
pd.DataFrame(rows)

,plan line,publisher,views (publisher count),"price per 1,000",① exact math (Decimal),same math as float,③ rounded to cents
0,PL-001,Instagram,"1,475,215",$8.25,12170.52375,12170.52375,"$12,170.52"
1,PL-002,Instagram,"2,204,466",$9.5,20942.427,20942.427,"$20,942.43"
2,PL-003,Instagram,"1,447,817",$12.5,18097.7125,18097.7125,"$18,097.71"
3,PL-004,Instagram,"10,678,674",$8.5,90768.729,90768.729,"$90,768.73"
4,PL-005,Instagram,"2,984,159",$8.25,24619.31175,24619.31175,"$24,619.31"
5,PL-006,Instagram,"1,169,556",$9.25,10818.393,10818.393,"$10,818.39"
6,PL-008,Instagram,"3,115,518",$9.25,28818.5415,28818.5415,"$28,818.54"
7,PL-009,Facebook,"4,577,110",$10.25,46915.3775,46915.3775,"$46,915.38"


## ② The rule

| Rule | Why | Small example |
|---|---|---|
| Money is always `Decimal`; a float is refused | A float can't store most cent amounts exactly | float 0.1 + 0.2 = 0.30000000000000004 |
| Round to whole cents, **half up** | The usual rule on invoices | \$10.125 → \$10.13 |
| Round **each line first**, then add | Every line on a bill matches the total | 3 × \$0.005 → \$0.01 each → total \$0.03 |

## ③ Half up vs the other ways of rounding

Python's built-in Decimal default is **banker's rounding** (a half goes to the nearest
*even* cent). Float `round()` is worse: it is sometimes wrong, because the float can't
hold the amount exactly (1.005 is really stored as 1.00499999…).

In [22]:
amounts = ["10.125", "10.135", "2.675", "1.005", "12170.52375"]
pd.DataFrame(
    [
        {
            "exact amount": a,
            "our rule: half up": f"${to_cents(Decimal(a))}",
            "banker's (half to even)": f"${Decimal(a).quantize(Decimal('0.01'))}",
            "float round()": f"${round(float(a), 2)}",
        }
        for a in amounts
    ]
)

,exact amount,our rule: half up,banker's (half to even),float round()
0,10.125,$10.13,$10.12,$10.12
1,10.135,$10.14,$10.14,$10.13
2,2.675,$2.68,$2.68,$2.67
3,1.005,$1.01,$1.00,$1.0
4,12170.52375,$12170.52,$12170.52,$12170.52


## Each line first, then the total

3 bill lines of 1 view each, at \$5 per 1,000 views = \$0.005 per line.

In [23]:
lines = [cost_of_views(1, Decimal("5")) for _ in range(3)]
exact_sum = 3 * Decimal("0.005")

print("each line, rounded:      ", [f"${x}" for x in lines])
print("total of rounded lines:  ", f"${total(lines)}")
print("rounding only the total: ", f"${to_cents(exact_sum)}  <- wouldn't match the lines")

try:
    total([Decimal("0.01"), Decimal("0.005")])
except ValueError as error:
    print("\n❌ refused:", error)

each line, rounded:       ['$0.01', '$0.01', '$0.01']
total of rounded lines:   $0.03
rounding only the total:  $0.02  <- wouldn't match the lines

❌ refused: round each line to cents before adding: got 0.005


## A float is refused

In [24]:
try:
    cost_of_views(100, 8.25)
except TypeError as error:
    print("❌ refused:", error)

❌ refused: price_per_1000_views must be a Decimal, not float


## A real bill's lines, rounded and added

All January delivery lines for Instagram, each rounded to cents, then added up.

In [25]:
instagram_january = [
    cost_of_views(d.publisher_count, ref.plan_lines[pid].price_per_1000_views)
    for (pid, month), d in ref.deliveries.items()
    if d.publisher == "Instagram" and month == "2026-01"
]
print(f"{len(instagram_january)} lines, total ${total(instagram_january):,}")

7 lines, total $206,235.63


---
**Next section, step 2.2:** read the Facebook bills (CSV) into `InvoiceLine` forms.